In [ ]:
# ──────────────────────────────────────────────
# STEP 1 – GPU cleanup
# ──────────────────────────────────────────────
import os, gc, time, json, warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import nibabel as nib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("GPU cleaned")

In [ ]:
# ──────────────────────────────────────────────
# STEP 2 – Device
# ──────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ──────────────────────────────────────────────
# STEP 3 – Paths
# ──────────────────────────────────────────────
PROJECT_ROOT = os.path.expanduser("/home/chest_ct/code")
DATA_ROOT    = os.path.join(PROJECT_ROOT, "data")
VOLUMES_DIR  = os.path.join(DATA_ROOT, "segmentations/segmentations")
REXCT_DIR    = os.path.join(DATA_ROOT, "rexgrounding-ct")
DATASET_JSON = os.path.join(REXCT_DIR, "dataset.json")
RESULTS_DIR  = os.path.join(PROJECT_ROOT, "models/merlin/results_merlin")
CACHE_DIR    = os.path.join(PROJECT_ROOT, "cache")

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR,   exist_ok=True)

In [ ]:
# ──────────────────────────────────────────────
# STEP 4 – Metadata
# ──────────────────────────────────────────────
with open(DATASET_JSON, "r") as f:
    metadata = json.load(f)

samples = metadata if isinstance(metadata, list) else list(metadata.values())[0]
print("Total samples:", len(samples))

In [ ]:
# ──────────────────────────────────────────────
# STEP 5 – Index CT files
# ──────────────────────────────────────────────
ct_index = {}
for root, dirs, files in os.walk(VOLUMES_DIR):
    for fname in files:
        if fname.endswith(".nii.gz"):
            ct_index[fname] = os.path.join(root, fname)

print("CT files indexed:", len(ct_index))

In [ ]:
# ──────────────────────────────────────────────
# STEP 6 – Match  (add [:1] here to test 1 scan)
# ──────────────────────────────────────────────
matched = [s for s in samples if s["name"] in ct_index][:1]
print("Matched:", len(matched))

In [ ]:
# ──────────────────────────────────────────────
# STEP 7 – Preprocessing helpers
# ──────────────────────────────────────────────
TARGET_SHAPE = (96, 96, 96)

def resize_ct_torch(ct_np, target=TARGET_SHAPE):
    t = torch.from_numpy(ct_np.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = torch.nn.functional.interpolate(t, size=target, mode="trilinear", align_corners=False)
    return t.squeeze(0)   # (1, D, H, W)


def load_and_preprocess(ct_path):
    img    = nib.load(ct_path)
    affine = img.affine
    ct     = np.asarray(img.dataobj, dtype=np.float32)
    while ct.ndim > 3:
        ct = ct.squeeze()
    if ct.ndim != 3:
        raise ValueError(f"Unexpected CT shape: {ct.shape}")
    ct = np.clip(ct, -1000.0, 400.0)
    ct = (ct + 1000.0) / 1400.0
    np.nan_to_num(ct, copy=False)
    return resize_ct_torch(ct), affine, ct.shape   # tensor, affine, orig_shape


In [ ]:
# ──────────────────────────────────────────────
# STEP 8 – Load embedding model
# ──────────────────────────────────────────────
from merlin import Merlin

print("Loading embedding model...")
model_embed = Merlin(ImageEmbedding=True).to(DEVICE).eval()
print("Embedding model loaded")

In [ ]:
# ──────────────────────────────────────────────
# STEP 9 – Batch embedding
# ──────────────────────────────────────────────
BATCH_SIZE = 4

results = []
errors  = []

print(f"\nRunning embedding phase (batch_size={BATCH_SIZE}) ...")
t0 = time.time()


def run_embedding_batch(batch_samples):
    tensors, metas = [], []
    for s in batch_samples:
        name    = s["name"]
        ct_path = ct_index[name]
        try:
            tensor, affine, orig_shape = load_and_preprocess(ct_path)
            tensors.append(tensor)
            metas.append({"name": name, "affine": affine,
                          "ct_path": ct_path, "orig_shape": orig_shape})
        except Exception as e:
            print(f"  [LOAD ERROR] {name}: {e}")
            metas.append(None)

    if not tensors:
        return [None] * len(batch_samples)

    batch_tensor = torch.stack(tensors, dim=0).to(DEVICE)
    try:
        with torch.inference_mode():
            with torch.amp.autocast(device_type="cuda"):
                embeddings = model_embed(batch_tensor)
        embeddings_np = embeddings.detach().cpu().numpy()
    except Exception as e:
        print(f"  [GPU ERROR] {e}")
        return [None] * len(batch_samples)
    finally:
        del batch_tensor
        torch.cuda.empty_cache()

    out, ei = [], 0
    for meta in metas:
        if meta is None:
            out.append(None)
        else:
            meta["embedding"] = embeddings_np[ei]
            out.append(meta)
            ei += 1
    gc.collect()
    return out


for batch_start in range(0, len(matched), BATCH_SIZE):
    batch      = matched[batch_start: batch_start + BATCH_SIZE]
    batch_outs = run_embedding_batch(batch)
    for s, out in zip(batch, batch_outs):
        if out is None:
            errors.append(s["name"])
        else:
            results.append(out)
    elapsed = time.time() - t0
    done    = batch_start + len(batch)
    eta     = (elapsed / done) * (len(matched) - done) if done else 0
    print(f"  [{done}/{len(matched)}]  success={len(results)}  "
          f"errors={len(errors)}  elapsed={elapsed/60:.1f}m  ETA={eta/60:.1f}m")

print(f"Embedding DONE in {(time.time()-t0)/60:.1f} min")


In [ ]:
# ──────────────────────────────────────────────
# STEP 10 – Load report generation model
# ──────────────────────────────────────────────

print("Loading report generation model...")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

from merlin import Merlin

model_report = Merlin(RadiologyReport=True)
model_report = model_report.to(DEVICE)
model_report.eval()

print("Report model loaded")

In [ ]:
# ──────────────────────────────────────────────
# STEP 11 – Target findings + GradCAM config
# ──────────────────────────────────────────────

TARGET_FINDINGS = [
    "lung_opacity",
    "atelectasis",
    "consolidation",
    "lung_nodule"
]

FINDING_KEYWORDS = {
    "lung_opacity": ["lung opacity", "opacity"],
    "atelectasis": ["atelectasis"],
    "consolidation": ["consolidation"],
    "lung_nodule": ["lung nodule", "nodule"]
}

GRADCAM_THRESHOLD = 0.35

In [ ]:
# ──────────────────────────────────────────────
# STEP 12 – Report generation + GradCAM
# ──────────────────────────────────────────────

REPORT_BATCH_SIZE = 1

TARGET_FINDINGS = [
    "lung_opacity",
    "atelectasis",
    "consolidation",
    "lung_nodule"
]

FINDING_KEYWORDS = {
    "lung_opacity": ["lung opacity", "opacity"],
    "atelectasis": ["atelectasis"],
    "consolidation": ["consolidation"],
    "lung_nodule": ["lung nodule", "nodule"]
}

GRADCAM_THRESHOLD = 0.35


# -------------------------------------------------
# Finding detector
# -------------------------------------------------

def finding_in_report(report_text, finding_key):

    if report_text is None:
        return False

    low = report_text.lower()

    return any(
        kw in low
        for kw in FINDING_KEYWORDS[finding_key]
    )


# -------------------------------------------------
# Save nifti helper
# -------------------------------------------------

def save_nifti(volume, affine, path):

    img = nib.Nifti1Image(
        volume.astype(np.float32),
        affine
    )

    nib.save(img, path)


# -------------------------------------------------
# Simple GradCAM
# -------------------------------------------------

def make_heatmap(ct_tensor):

    with torch.no_grad():

        feat = model_embed(ct_tensor)

        if isinstance(feat, (list, tuple)):
            feat = feat[0]

        heat = feat.mean(dim=1)

        heat = torch.relu(heat)

        heat = heat.squeeze().cpu().numpy()

        heat = (
            heat - heat.min()
        ) / (
            heat.max() - heat.min() + 1e-8
        )

    return heat


print("\nRunning report generation...")


for idx, r in enumerate(results):

    print(f"\n[{idx+1}/{len(results)}] {r['name']}")

    try:

        # -------------------------------------------------
        # Reload CT
        # -------------------------------------------------

        tensor, affine, orig_shape = load_and_preprocess(
            r["ct_path"]
        )

        tensor = tensor.unsqueeze(0).to(DEVICE)

        print("CT loaded")


        # -------------------------------------------------
        # Generate report
        # -------------------------------------------------

        with torch.no_grad():

            report_out = model_report(tensor)

        print("Report generated")


        # -------------------------------------------------
        # Convert report output to string
        # -------------------------------------------------

        if isinstance(report_out, str):

            report_text = report_out

        elif isinstance(report_out, list):

            report_text = str(report_out[0])

        elif isinstance(report_out, dict):

            if "report" in report_out:

                report_text = str(
                    report_out["report"]
                )

            elif "text" in report_out:

                report_text = str(
                    report_out["text"]
                )

            else:

                report_text = str(report_out)

        else:

            report_text = str(report_out)


        r["report"] = report_text


        # -------------------------------------------------
        # Detect findings
        # -------------------------------------------------

        detected = []

        for finding in TARGET_FINDINGS:

            if finding_in_report(
                report_text,
                finding
            ):

                detected.append(finding)

        r["present_findings"] = detected

        print("Findings:", detected)


        # -------------------------------------------------
        # Create output directory
        # -------------------------------------------------

        scan_dir = os.path.join(
            RESULTS_DIR,
            r["name"].replace(".nii.gz", "")
        )

        os.makedirs(
            scan_dir,
            exist_ok=True
        )


        # -------------------------------------------------
        # Save report
        # -------------------------------------------------

        report_path = os.path.join(
            scan_dir,
            "report.txt"
        )

        with open(report_path, "w") as f:

            f.write(report_text)


        # -------------------------------------------------
        # Generate heatmap
        # -------------------------------------------------

        heatmap = make_heatmap(tensor)

        print("Heatmap generated")


        # -------------------------------------------------
        # Resize heatmap
        # -------------------------------------------------

        heat_tensor = torch.from_numpy(
            heatmap
        ).unsqueeze(0).unsqueeze(0)

        heat_tensor = torch.nn.functional.interpolate(
            heat_tensor,
            size=orig_shape,
            mode="trilinear",
            align_corners=False
        )

        heatmap_resized = (
            heat_tensor.squeeze().numpy()
        )


        # -------------------------------------------------
        # Binary mask
        # -------------------------------------------------

        mask = (
            heatmap_resized > GRADCAM_THRESHOLD
        ).astype(np.uint8)


        # -------------------------------------------------
        # Save nifti outputs
        # -------------------------------------------------

        heatmap_path = os.path.join(
            scan_dir,
            "heatmap.nii.gz"
        )

        mask_path = os.path.join(
            scan_dir,
            "mask.nii.gz"
        )

        save_nifti(
            heatmap_resized,
            affine,
            heatmap_path
        )

        save_nifti(
            mask,
            affine,
            mask_path
        )


        # -------------------------------------------------
        # Load original CT for preview
        # -------------------------------------------------

        ct_np = nib.load(
            r["ct_path"]
        ).get_fdata()

        z = ct_np.shape[2] // 2


        # -------------------------------------------------
        # Save preview image
        # -------------------------------------------------

        fig = plt.figure(figsize=(12, 4))

        gs = gridspec.GridSpec(1, 3)

        # CT
        ax1 = fig.add_subplot(gs[0])

        ax1.imshow(
            ct_np[:, :, z],
            cmap="gray"
        )

        ax1.set_title("CT")

        ax1.axis("off")


        # Heatmap
        ax2 = fig.add_subplot(gs[1])

        ax2.imshow(
            heatmap_resized[:, :, z],
            cmap="hot"
        )

        ax2.set_title("Heatmap")

        ax2.axis("off")


        # Overlay
        ax3 = fig.add_subplot(gs[2])

        ax3.imshow(
            ct_np[:, :, z],
            cmap="gray"
        )

        ax3.imshow(
            heatmap_resized[:, :, z],
            cmap="jet",
            alpha=0.4
        )

        ax3.set_title("Overlay")

        ax3.axis("off")


        png_path = os.path.join(
            scan_dir,
            "preview.png"
        )

        plt.tight_layout()

        plt.savefig(
            png_path,
            bbox_inches="tight"
        )

        plt.close()

        print("Preview saved")


    except Exception as e:

        print("ERROR:", e)

        r["report"] = None

        r["present_findings"] = []

        continue


print("\nDONE")

In [ ]:

# ============================================================
# STEP 13 — REPORT INFERENCE + 3D GRAD-CAM
# ============================================================

import os
import time
import numpy as np
import torch
import torch.nn.functional as F

# ============================================================
# 1. DEFINE 3D GRADCAM CLASS
# ============================================================

class GradCAM3D:

    def __init__(self, model):

        self.model = model
        self.gradients = None
        self.activations = None

        # ----------------------------------------------------
        # Try to hook the LAST Conv3D layer automatically
        # ----------------------------------------------------
        target_layer = None

        for m in reversed(list(model.modules())):
            if isinstance(m, torch.nn.Conv3d):
                target_layer = m
                break

        if target_layer is None:
            raise ValueError("No Conv3d layer found in model")

        self.target_layer = target_layer

        # Forward hook
        self.target_layer.register_forward_hook(
            self.save_activation
        )

        # Backward hook
        self.target_layer.register_full_backward_hook(
            self.save_gradient
        )

        print("\nGradCAM hooked layer:")
        print(self.target_layer)

    # --------------------------------------------------------
    # Save feature maps
    # --------------------------------------------------------
    def save_activation(self, module, inp, output):
        self.activations = output.detach()

    # --------------------------------------------------------
    # Save gradients
    # --------------------------------------------------------
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    # --------------------------------------------------------
    # Generate CAM
    # --------------------------------------------------------
    def generate(self, x, class_idx):

        self.model.zero_grad()

        output = self.model(x)

        # Single class score
        score = output[:, class_idx]

        score.backward(retain_graph=True)

        grads = self.gradients
        acts = self.activations

        # Global average pooling
        weights = grads.mean(dim=(2, 3, 4), keepdim=True)

        cam = (weights * acts).sum(dim=1)

        cam = F.relu(cam)

        cam = cam.squeeze().cpu().numpy()

        # Normalize
        cam = cam - cam.min()

        if cam.max() > 0:
            cam = cam / cam.max()

        return cam


# ============================================================
# 2. LABELS
# ============================================================

TARGET_DISEASES = [
    "lung opacity",
    "atelectasis",
    "consolidation",
    "lung nodule"
]

# ============================================================
# 3. SAFE FORWARD FUNCTION
# ============================================================

def forward_report_model(model, volume_tensor):

    with torch.cuda.amp.autocast():

        out = model(volume_tensor)

        # Handle different output formats
        if isinstance(out, dict):

            if "logits" in out:
                logits = out["logits"]

            elif "pred_logits" in out:
                logits = out["pred_logits"]

            else:
                logits = list(out.values())[0]

        else:
            logits = out

    return logits


# ============================================================
# 4. PROCESS SINGLE SCAN
# ============================================================

def process_report(sample):

    try:

        volume = sample["volume"]

        # ---------------------------------------------
        # Tensor shape:
        # [1, 1, D, H, W]
        # ---------------------------------------------
        x = torch.tensor(
            volume,
            dtype=torch.float32
        ).unsqueeze(0).unsqueeze(0).cuda()

        # ---------------------------------------------
        # Forward pass
        # ---------------------------------------------
        with torch.no_grad():

            logits = forward_report_model(
                model_report,
                x
            )

            probs = torch.sigmoid(logits)[0]

        probs = probs.detach().cpu().numpy()

        # ---------------------------------------------
        # Disease predictions
        # ---------------------------------------------
        findings = []

        for i, disease in enumerate(TARGET_DISEASES):

            if probs[i] > 0.5:

                findings.append({
                    "disease": disease,
                    "probability": float(probs[i])
                })

        # ---------------------------------------------
        # GradCAM generation
        # ---------------------------------------------
        gradcams = {}

        for i, disease in enumerate(TARGET_DISEASES):

            if probs[i] > 0.5:

                cam = gradcam.generate(x, i)

                gradcams[disease] = cam

        return {
            "name": sample["name"],
            "findings": findings,
            "gradcam": gradcams
        }

    except Exception as e:

        print(f"\nERROR: {sample['name']}")
        print(e)

        return None


# ============================================================
# 5. INITIALISE GRADCAM
# ============================================================

gradcam = GradCAM3D(model_report)

# ============================================================
# 6. RUN REPORT PHASE
# ============================================================

report_results = []
report_errors = []

print("\nRunning report + GradCAM phase...")
t1 = time.time()

for i, sample in enumerate(results):

    out = process_report(sample)

    if out is None:
        report_errors.append(sample["name"])
    else:
        report_results.append(out)

    print(
        f"[{i+1}/{len(results)}] "
        f"success={len(report_results)} "
        f"errors={len(report_errors)}"
    )

t2 = time.time()

print("\n===================================")
print("REPORT PHASE COMPLETE")
print("===================================")

print(f"Successful : {len(report_results)}")
print(f"Errors     : {len(report_errors)}")
print(f"Time       : {(t2-t1)/60:.2f} min")


In [ ]:
# ──────────────────────────────────────────────
# STEP 14 – Filter results to 4 target findings
# ──────────────────────────────────────────────
filtered_results = [r for r in results if r.get("present_findings")]

print(f"\nFiltered: {len(filtered_results)}/{len(results)} scans have target findings")

In [ ]:
# ──────────────────────────────────────────────
# STEP 15 – Save global JSON outputs
# ──────────────────────────────────────────────
def serialise(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f"Not serialisable: {type(obj)}")


all_out      = os.path.join(RESULTS_DIR, "final_results.json")
filtered_out = os.path.join(RESULTS_DIR, "filtered_results.json")
errors_out   = os.path.join(RESULTS_DIR, "errors.json")

with open(all_out, "w") as f:
    json.dump(results, f, indent=2, default=serialise)
print("Saved:", all_out)

with open(filtered_out, "w") as f:
    json.dump(filtered_results, f, indent=2, default=serialise)
print("Saved:", filtered_out)

with open(errors_out, "w") as f:
    json.dump(errors, f, indent=2)
print("Saved:", errors_out)

In [ ]:

# ──────────────────────────────────────────────
# STEP 16 – Print output tree
# ──────────────────────────────────────────────
print("\n── Output tree ──────────────────────────────────────")
for r in filtered_results:
    scan_dir = os.path.join(RESULTS_DIR, r["name"].replace(".nii.gz", ""))
    print(f"\n{r['name']}  findings={r['present_findings']}")
    if os.path.isdir(scan_dir):
        for fname in sorted(os.listdir(scan_dir)):
            print(f"  {fname}")

print("\nPipeline complete ✓")